# 🔧 Function Calling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nawooooojin/openai-api-tutorial-kr/blob/main/1_Text_Generation/03-Function%20Calling-Lecture.ipynb)

> **주의:** Colab 런타임을 재시작하면 환경 변수가 초기화되므로 `OPENAI_API_KEY`를 다시 설정해야 합니다.

## 1) 환경 준비 및 API 키 설정

In [ ]:
%pip install -q -U openai requests

### API 키 설정 및 Client 생성
- **Colab**: 왼쪽 메뉴 `보안 비밀(열쇠 아이콘)`에 `OPENAI_API_KEY`를 저장하고 **노트북 액세스**를 켜 두면 자동으로 불러옵니다.
- **로컬**: 환경 변수 `OPENAI_API_KEY`를 미리 설정해 두거나, 아래 셀을 실행했을 때 뜨는 입력창에 키를 붙여넣으면 됩니다.
- 아래의 코드에서 오류가 난다면 API 키의 오류일 가능성이 높습니다.

In [1]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata  # type: ignore
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except ImportError:
        pass  # Colab이 아닌 환경
    except Exception:
        print("Colab 메뉴의 보안 비밀(열쇠 아이콘)에 OPENAI_API_KEY가 저장되어 있는지, 노트북 액세스가 켜져 있는지 확인해주세요.")

if not OPENAI_API_KEY:
    from getpass import getpass
    OPENAI_API_KEY = getpass("OPENAI_API_KEY를 입력하세요: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("✅ API 키 설정 완료")

✅ API 키 설정 완료


In [2]:
import json
from openai import OpenAI

client = OpenAI()
print("✅ 클라이언트 생성 완료")

✅ 클라이언트 생성 완료


## 2) Function Calling이란?

모델이 외부 기능(툴/함수)을 사용할 수 있게 해주는 기능입니다.

이를 통해 모델은 훈련 데이터에 포함되지 않은 실시간 정보나 내부 시스템에 접근할 수 있습니다.

* **주요 활용 예시:**
    * 실시간 날씨 정보 조회
    * 사용자 데이터베이스(계정 정보) 조회
    * 주문 취소 및 환불 처리 등 액션 수행

### Function Calling의 구성 요소

* **Tool (도구)**
    * 개발자가 모델에게 "이런 기능이 있다"고 제공하는 정보입니다.
    * 주로 `JSON Schema`로 정의된 function 또는 custom tool 형식을 따릅니다.
* **Tool Call (도구 호출)**
    * 모델이 사용자의 질문을 분석한 뒤, 실제로 "이 툴을 써야겠다"고 판단하여 보내는 **특별한 응답**입니다.
    * 실행해야 할 함수 이름과 인자(arguments)를 포함합니다.
* **Tool Call Output (실행 결과)**
    * 실제 애플리케이션(코드)에서 툴을 실행하고 나온 결과값입니다.
    * JSON 또는 Text 형태이며, 모델은 이 결과를 바탕으로 최종 답변을 생성합니다.

### 🔄 Function Calling의 5단계 흐름

1. **Request**: 유저의 질문과 함께 **사용 가능한 도구(Tools)의 목록**을 모델에게 보냅니다.
2. **Tool Call**: 모델이 질문을 분석하고 도구 사용이 필요하면, **함수 이름과 인자(Arguments)** 를 응답으로 보냅니다.
3. **Execute**: 실제 코드(Python 등)에서 해당 함수를 실행하여 결과를 얻습니다. (LLM 모델이 실행하는 게 아닙니다)
4. **Submit**: 함수의 실행 결과(Output)를 다시 모델에게 전달합니다. (History에 추가)
5. **Response**: 모델이 함수 결과를 바탕으로 최종 자연어 답변을 생성합니다.

## 3) Function Calling 실습

### Step 0: 도구(Tool) 정의 및 함수 구현하기

모델에게 도구를 설명할 때는 `JSON Schema` 형식을 사용합니다.

함수 이름, 설명, 파라미터 타입 등을 명확히 적어야 모델이 올바르게 tool을 사용할 수 있습니다.

| FIELD | DESCRIPTION |
|-------|-------------|
| type | 항상 "function"으로 설정 |
| name | 함수 이름 (예: get_weather) |
| description | 함수 사용 시점 및 방법에 대한 설명 |
| parameters | 함수 입력 인자를 정의하는 JSON 스키마 |
| strict | 함수 호출에 대한 strict mode 적용 여부 |

아래는 예시인 `get_weather` 함수의 정의입니다.

In [3]:
# 1. 모델이 사용할 도구(함수) 정의
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "특정 위치의 현재 날씨를 조회합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "도시 및 국가 (예: Seoul, Korea)"
                },
                "units": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "온도 단위"
                }
            },
            "required": ["location", "units"],
            "additionalProperties": False
        },
        "strict": True  # Strict Mode 활성화 (스키마 준수 강제)
    }
]

print("✅ 도구 정의 완료")

✅ 도구 정의 완료


### Best Practices

정확도를 높이기 위해 다음 원칙을 따르는 것이 좋습니다.

1.  **명확한 설명 (Description):**
    * 함수의 목적, 각 매개 변수의 포맷이 무엇인지, 출력은 무엇인지 구체적으로 작성하세요.
    * System 프롬프트를 사용하여 각 기능을 언제 사용해야 하는지 명시하면 더 좋습니다.
2.  **소프트웨어 엔지니어링 원칙 적용:**
    * 함수를 명확하고 직관적으로 만드세요.
    * `enum`을 사용하여 유효하지 않은 값(Invalid state)이 들어오지 못하게 막으세요.
3.  **모델의 부담 줄이기:**
    * 모델이 이미 알고 있는 정보를 인자로 요구하지 마세요.
    * 항상 같이 쓰이는 함수들은 하나로 합치세요.
4.  **함수 개수 제한:**
    * 한 번에 너무 많은 함수를 제공하면 성능이 저하될 수 있습니다 (20개 미만 권장).
5.  **Playground에서 테스트하기:**
    * Playground에서 스키마를 테스트하고 튜닝하세요.
        - Playground: https://platform.openai.com/chat

### get_weather 함수 구현

In [4]:
# function 구현
def get_weather(location, units="celsius"):

    if "seoul" in location.lower():
        return json.dumps({"temperature": "22", "unit": units, "description": "맑음"}, ensure_ascii=False)
    elif "paris" in location.lower():
        return json.dumps({"temperature": "15", "unit": units, "description": "흐림"}, ensure_ascii=False)
    else:
        return json.dumps({"temperature": "unknown"})

### Step 1 & 2: 질문하기 및 Tool Call 수신
모델에게 질문을 던지면, 모델은 답변 텍스트 대신 `function_call` 객체를 반환할 것입니다.
- `response.output`에는 `reasoning` 아이템 등이 같이 들어올 수 있어서, 몇 번째에 있는지 가정하지 말고 `type`으로 찾아야 합니다.

In [5]:
input_messages = [
    {"role": "user", "content": "지금 서울 날씨가 어때?"}
]

response = client.responses.create(
    model="gpt-6-luna",
    tools=tools,
    input=input_messages,
)

# 응답 확인
for item in response.output:
    print("Output Type:", item.type)
print("Full Output:", response.output)

Output Type: function_call
Full Output: [ResponseFunctionToolCall(arguments='{"location":"Seoul, Korea","units":"celsius"}', call_id='call_qdeM5LLkeHQ6uvbrxN7CMEdT', name='get_weather', type='function_call', id='fc_09617bebc057a8eb006adf6707654487d2a34eed27afd0068f', async_=None, caller=None, namespace=None, status='completed')]


### Step 3 & 4: 함수 실행 및 결과 전달
모델이 `get_weather`를 호출해달라고 요청했으므로, 우리는 코드를 실행하고 그 결과를 다시 모델에게 알려줘야 합니다.


**모델의 응답(`output`) 처리 로직**

1.  **감지:** `output` 배열 안에 `type="function_call"`인 요소가 있는지 확인합니다.
2.  **파싱:** 각 요소의 `call_id`, `name`, `arguments`(JSON 문자열)를 추출합니다.
3.  **실행:** `arguments`를 `json.loads`하여 실제 Python/JS 함수를 실행합니다.
4.  **결과 생성:** 실행 결과를 `function_call_output` 타입의 메시지로 만들어 리스트에 추가(`append`)합니다.
    - 반드시 문자열(String) type
    - 구조화된 데이터는 JSON String 권장
    - 반환값이 없는 함수라도 "success" 등의 단순 문자열 반환 필수

In [6]:
# 모델의 응답을 대화 기록에 추가
input_messages += response.output

# Tool Call 처리 루프
for item in response.output:
    if item.type == "function_call":
        name = item.name
        arguments = json.loads(item.arguments)
        call_id = item.call_id

        print(f"🛠️ 함수 호출 감지: {name}, 인자: {arguments}")

        if name == "get_weather":
            # 함수 실행
            result = get_weather(location=arguments["location"], units=arguments["units"])

            # 결과 메시지 생성
            tool_output_message = {
                "type": "function_call_output",
                "call_id": call_id,
                "output": result
            }

            # 대화 기록에 결과 추가
            input_messages.append(tool_output_message)
            print(f"📤 tool 결과 : {result}")

🛠️ 함수 호출 감지: get_weather, 인자: {'location': 'Seoul, Korea', 'units': 'celsius'}
📤 tool 결과 : {"temperature": "22", "unit": "celsius", "description": "맑음"}


### Step 5: 최종 응답 받기
함수 실행 결과(`function_call_output`)가 포함된 대화 기록을 모델에게 다시 보내면, 모델은 이를 해석하여 자연어 답변을 내놓습니다.

In [7]:
final_response = client.responses.create(
    model="gpt-6-luna",
    tools=tools,
    input=input_messages,
)

print("🤖 최종 답변:", final_response.output_text)

🤖 최종 답변: 서울은 현재 맑고, 기온은 약 22°C예요.


## 4) 전체 파이프라인 자동화 예제 (Horoscope)

In [8]:
# 1. 도구 정의
horoscope_tools = [
    {
        "type": "function",
        "name": "get_horoscope",
        "description": "별자리에 대한 오늘의 운세를 조회합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "sign": {
                    "type": "string",
                    "description": "별자리 (예: Aquarius, Taurus 등)",
                },
            },
            "required": ["sign"],
        },
    },
]

# 2. Python 함수 구현
def get_horoscope(sign):
    return f"{sign}: 다음 주 화요일에 수달과 친구가 될 운명입니다!"

# 3. 자동화된 대화 처리
input_list = [
    {"role": "user", "content": "나는 물병자리(Aquarius)야. 내 운세를 알려줘."}
]

# 첫 번째 호출 (도구 탐색)
response = client.responses.create(
    model="gpt-6-luna",
    tools=horoscope_tools,
    input=input_list,
)

input_list += response.output # 모델의 응답 저장

# Tool Call 처리
for item in response.output:
    if item.type == "function_call":
        if item.name == "get_horoscope":
            # 인자 파싱 및 실행
            args = json.loads(item.arguments)
            result = get_horoscope(args["sign"])

            # 결과 메시지 추가
            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({"horoscope": result})
            })

# 두 번째 호출 (결과 반영 후 최종 답변 생성)
final_response = client.responses.create(
    model="gpt-6-luna",
    instructions="도구에서 생성된 운세 정보만 사용하여 답변하세요.",
    tools=horoscope_tools,
    input=input_list,
)

print("🔮 운세 결과:", final_response.output_text)

🔮 운세 결과: 다음 주 화요일에는 수달과 친구가 될 운명이에요!


## 5) `tool_choice` 옵션
모델이 응답을 생성할 때 어떤 툴들을 사용할지 선택하는 방식을 지정합니다.

* `auto` (기본값): 모델이 알아서 0개~여러 개의 함수를 호출할지 결정합니다.
    - `tool_choice = "auto"`
* `required`: 반드시 1개 이상의 함수를 호출하도록 강제합니다.
    - `tool_choice = "required"`
* `forced function`: 특정 함수 하나를 호출하도록 강제합니다.
    - `tool_choice = {"type": "function", "name": "get_weather"}`
* `none`: 함수가 정의되어 있어도 사용하지 않도록 강제합니다.
    - `tool_choice = "none"`
* `allowed_tools`: 모델이 사용 가능한 함수 중 특정 함수들만 사용하도록 제한합니다. (Prompt Caching 효율 유지에 유리)

In [9]:
# allowed_tools 예시
get_weather_tool = tools[0]  # 위에서 정의한 get_weather

search_docs_tool = {
    "type": "function",
    "name": "search_docs",
    "description": "사내 문서에서 키워드로 관련 문서를 검색합니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "검색어"}
        },
        "required": ["query"],
        "additionalProperties": False,
    },
    "strict": True,
}

web_search_tool = {"type": "web_search"}

response = client.responses.create(
    model="gpt-6-luna",
    input="휴가 규정 문서를 찾아줘. 날씨는 무시해도 좋아.",
    tools=[get_weather_tool, search_docs_tool, web_search_tool],  # 툴 목록은 그대로 두고
    tool_choice={                                                  # 이번 요청에서 쓸 수 있는 툴만 제한
        "type": "allowed_tools",
        "mode": "auto",
        "tools": [
            {"type": "function", "name": "search_docs"},
            {"type": "function", "name": "get_weather"}
        ]
    }
)

for item in response.output:
    if item.type == "function_call":
        print("🛠️ 호출된 함수:", item.name, item.arguments)

🛠️ 호출된 함수: search_docs {"query":"휴가 규정 문서"}


### 참고 자료
- https://platform.openai.com/docs/guides/function-calling